In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_1756/3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [4]:
from google.colab import files

uploaded = files.upload()

Saving constitution.pdf to constitution.pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_file(data):
    loader = PyPDFLoader(data)
    documents = loader.load()
    return documents

In [7]:
import os
print(os.path.exists("constitution.pdf"))

True


In [8]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf

In [9]:
extracted_data = load_pdf_file("constitution.pdf")

In [10]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [11]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 2047


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

In [13]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [14]:
embeddings=download_hugging_face_embeddings()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
query_result=embeddings.embed_query("This is my first project")
print("Length",len(query_result))

Length 384


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")
pineconne_api_key=os.getenv("PINECONE_API_KEY")

In [35]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [36]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "constition-chatbot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully.")
else:
    print("Index already exists. Using existing index.")

Index already exists. Using existing index.


In [38]:

# Embed each chunk and upsert the embeddings into your Pinecone index.
# PineconeVectorStore: Connects LangChain with Pinecone to store and search document embeddings.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [39]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [40]:
docsearch

In [41]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [42]:
retrieved_docs = retriever.invoke("What is right to freedom")

In [43]:
retrieved_docs

[Document(id='49cf5307-11f9-4ef0-9814-7c6de12f7920', metadata={'creationdate': '', 'creator': 'PyPDF', 'moddate': '2024-07-01T11:20:33+00:00', 'page': 39.0, 'page_label': '40', 'producer': 'iLovePDF', 'source': 'constitution.pdf', 'total_pages': 402.0}, page_content='(4) No person holding any office of profit or trust under the State shall, \nwithout the consent of the President, accept any present, emolument, or office \nof any kind from or under any foreign State.\nRight to Freedom\n19. Protection of certain rights regarding freedom of speech, etc. —\n(1) All citizens shall have the right—\n(a) to freedom of speech and expression;\n(b) to assemble peaceably and without arms;\n(c) to form associations or unions2[or co-operative societies];'),
 Document(id='2b9534be-0ccf-4761-b0db-500dc1a051a9', metadata={'creationdate': '', 'creator': 'PyPDF', 'moddate': '2024-07-01T11:20:33+00:00', 'page': 44.0, 'page_label': '45', 'producer': 'iLovePDF', 'source': 'constitution.pdf', 'total_pages': 

In [44]:
from langchain_groq import ChatGroq
llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0
)

In [45]:
# ChatPromptTemplate: Creates a structured prompt template for sending
# system instructions and user messages to the LLM.
from langchain_core.prompts import ChatPromptTemplate
# RunnablePassthrough: Passes the input data through a LangChain chain
# without changing it.
from langchain_core.runnables import RunnablePassthrough
# StrOutputParser: Converts the LLM's response into a simple string.
from langchain_core.output_parsers import StrOutputParser


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "Context:\n{context}"
)

# Creates a chat prompt containing a system message and a human/user message.
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [46]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [47]:
response = rag_chain.invoke("If the Government imposes greater control and censorship over social media platforms to curb fake news, which fundamental rights under Article 19(1)(a), Article 19(2), and Article 21 (Right to Privacy from Puttaswamy judgment) will come into conflict, and how did the Supreme Court in Shreya Singhal v. Union of India resolve this balance?")
print(response)

Greater control over social media would pit **Article 19(1)(a)** (freedom of speech and expression) against **Article 19(2)** (the state’s power to impose reasonable restrictions) and **Article 21** (the right to privacy, as held in *Puttaswamy*). In *Shreya Singhal v. Union of India*, the Supreme Court struck down Section 66A of the IT Act, ruling that it was vague, overbroad and violated Article 19(1)(a); it held that any restriction must be narrowly tailored, clear, and proportionate, thereby preserving freedom of expression while allowing only reasonable, non‑arbitrary limits that do not infringe the privacy right.


In [55]:
import gradio as gr

def chatbot(question):
    if not question or not question.strip():
        return "Please enter a question."

    try:
        # 1. Search Pinecone using the question as a STRING
        docs = retriever.invoke(question)

        # 2. Get text from retrieved documents
        context = "\n\n".join(
            doc.page_content for doc in docs
        )

        # 3. Ask Groq using the retrieved Constitution content
        prompt = f"""
You are Samvidhan AI, a chatbot based on the Constitution of India.

Answer the user's question using ONLY the context provided below.
If the answer is not present in the context, say:
"I could not find this information in the provided Constitution document."

Give a clear and accurate answer.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

        response = llm.invoke(prompt)

        # 4. Return Groq's answer
        return response.content

    except Exception as e:
        return "Error: " + str(e)


demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Ask about Indian Constitution",
        placeholder="Example: Explain any one Fundamental Right"
    ),
    outputs=gr.Textbox(
        label="AI Answer"
    ),
    title="Samvidhan AI - Constitution of India Chatbot",
    description="Ask questions about Articles, Fundamental Rights, Directive Principles, etc."
)

demo.launch(share=True)





Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cc4f4bb5b2fe7fd5c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
